In [1]:
import torch
import pickle
import gzip
import os
from torch.utils.data import DataLoader
from datetime import datetime
import gc
from typing import Iterator
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import re

import sys
sys.path.append("../")

from shared_utils.data import CSVPromptDataset
from shared_utils.load import get_tokenizer, configs_from_yaml
from shared_utils.generate import generate_text

from early_exit.util import get_model

from early_exit.rewards import extract_solution

from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode

In [2]:
device = "cuda"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
model_config_path = "../config_deepseek.yaml"
#dataset_path = "../results_and_data/early_exit_sft_dataset/test/data_deduplicated.csv"
#prompt_config_path = "../results_and_data/early_exit_sft_dataset/test/prompt_config.json"
batch_size = 1
chunk_size = 10 #for saving data

In [3]:
output_dir = "teacher_generated_data"
#output_dir = "/workspace/data/teacher_generated_data_gzip"
os.makedirs(output_dir, exist_ok=True)

In [4]:
tokenizer = get_tokenizer(model_name)
config = configs_from_yaml(model_config_path, tokenizer.eos_token_id)
model = get_model(model_name, config['model'], device)

In [5]:
SYSTEM_PROMPT = "I am going to give you a math word problem. Solve it step by step, showing your reasoning. After your work, provide your final numerical answer"
PREFILLER = ""

In [6]:
#dataset = CSVPromptDataset(dataset_path, prompt_config_path)
#dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=dataset.collate_fn, shuffle=False)
def custom_collate_fn(batch):
    # Since batch_size=1, batch will contain only one PromptBatch object
    return batch[0]

def load_gsm8k_with_difficulty():
    gsm8k_dataset = load_dataset("gsm8k", "main")
    
    difficulty_dataset = load_dataset("lime-nlp/GSM8K_Difficulty", 'Difficulty Score')
    
    difficulty_df = pd.DataFrame(difficulty_dataset['train'])
    
    def categorize_difficulty(score):
        if score > 90:
            return "Easy"
        elif score >= 70:
            return "Medium"
        else:
            return "Hard"
    
    difficulty_df['difficulty_category'] = difficulty_df['solved_percentage'].apply(categorize_difficulty)
    difficulty_lookup = dict(zip(difficulty_df['problem'], difficulty_df[['solved_percentage', 'difficulty_category']].to_dict('records')))
    
    return gsm8k_dataset, difficulty_lookup

gsm8k_dataset, difficulty_lookup = load_gsm8k_with_difficulty()
data = gsm8k_dataset['train']

class GSM8KDataset:
    def __init__(self, data, difficulty_lookup):
        self.data = data
        self.difficulty_lookup = difficulty_lookup
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        question = item["question"]
        ground_truth_answer = item["answer"]  # Add this line
        difficulty_info = self.difficulty_lookup.get(question, {'solved_percentage': None, 'difficulty_category': 'Unknown'})
        
        class PromptBatch:
            def __init__(self, question, idx, difficulty_info, ground_truth_answer):
                self.full_user_prompt = [question]
                self.idx = [idx]
                self.difficulty = difficulty_info['solved_percentage']
                self.difficulty_category = difficulty_info['difficulty_category']
                self.ground_truth_answer = ground_truth_answer  # Add this line
        
        return PromptBatch(question, idx, difficulty_info, ground_truth_answer)

dataset = GSM8KDataset(data, difficulty_lookup)
dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=custom_collate_fn, shuffle=False)

In [7]:
model = replace_attention_layers(model, config['lora'], device)
model.eval() #not training

replacing layer model.layers.0
replacing layer model.layers.1
replacing layer model.layers.2
replacing layer model.layers.3
replacing layer model.layers.4
replacing layer model.layers.5
replacing layer model.layers.6
replacing layer model.layers.7
replacing layer model.layers.8
replacing layer model.layers.9
replacing layer model.layers.10
replacing layer model.layers.11
replacing layer model.layers.12
replacing layer model.layers.13
replacing layer model.layers.14
replacing layer model.layers.15
replacing layer model.layers.16
replacing layer model.layers.17
replacing layer model.layers.18
replacing layer model.layers.19
replacing layer model.layers.20
replacing layer model.layers.21
replacing layer model.layers.22
replacing layer model.layers.23
replacing layer model.layers.24
replacing layer model.layers.25
replacing layer model.layers.26
replacing layer model.layers.27
address this hack!
trainable params: 4,358,144 || all params: 1,913,738,780 || trainable%: 0.2277


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): DynamicallyTypedModelWithReadout(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x DynamicallyTypedLayerWithExit(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (early_exiter): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (early_exiter): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (early_exiter): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
         

In [8]:
all_teacher_data = []

In [9]:
current_chunk_data = []
chunk_idx = 0
total_samples_processed = 0

In [10]:
metadata = {
    'model_name': model_name,
    'dataset': 'gsm8k',
    'dataset_split': 'train',
    'config': config,
    'chunk_size': chunk_size,
    'timestamp': datetime.now().isoformat(),
    'system_prompt': SYSTEM_PROMPT,
    'prefiller': PREFILLER
}
metadata_path = os.path.join(output_dir, "metadata.pkl.gz")
with gzip.open(metadata_path, 'wb', compresslevel=9) as f:
    pickle.dump(metadata, f, protocol=pickle.HIGHEST_PROTOCOL)

In [11]:
def save_chunk(chunk_data, chunk_index, output_directory):
    """Save a chunk of data to disk with gzip compression"""
    chunk_filename = os.path.join(output_directory, f"chunk_{chunk_index:04d}.pkl.gz")

    #convert all float32 tensors to float16 before saving for size
    compressed_data = []
    for sample in chunk_data:
        compressed_sample = {}
        for key, value in sample.items():
            if isinstance(value, torch.Tensor) and value.dtype == torch.float32:
                if key == 'sft_teacher_final_layer_logprobs':
                    compressed_sample[key] = (value * (value > -14.0)).to_sparse().half() #switched to a sparse tensor and converts to float16
                else:
                    compressed_sample[key] = value.half() #converts to float16
            else:
                compressed_sample[key] = value
        compressed_data.append(compressed_sample)
    
    with gzip.open(chunk_filename, 'wb', compresslevel=9) as f: #play with compresslevel, 6 is moderate
        pickle.dump({
            'chunk_idx': chunk_index,
            'data': compressed_data,
            'num_samples': len(chunk_data)
        }, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    file_size_mb = os.path.getsize(chunk_filename) / (1024 * 1024)
    print(f"Saved chunk {chunk_index} with {len(chunk_data)} samples")
    print(f"  File: {chunk_filename}")
    print(f"  Size: {file_size_mb:.2f} MB")

In [12]:
def extract_answer_qwen(solution_str):
    """
    Extract numerical answer from Qwen model output that uses \\boxed{} format
    """
    #\boxed{} format (Qwen's preferred format)
    solutions = re.findall(r"\\boxed\{([+-]?[0-9\\.\\,]+)\}", solution_str)
        
    #fallback for **Final Answer:** format
    if len(solutions) == 0:
        solutions = re.findall(r"\*\*Final Answer:\*\*[\\s\\n]*([+-]?[0-9\\.\\,]+)", solution_str)
        
    if len(solutions) == 0:
        final_answer = None
    else:
        final_answer = solutions[-1].replace(",", "").replace("$", "")
    
    return final_answer

In [ ]:
for batch_idx, prompt_batch in enumerate(dataloader):
    if total_samples_processed >= 30:
        break
    if total_samples_processed % 50 == 0:
        print(f"Processing batch {total_samples_processed + 1}/{len(dataloader)} (Total samples: {total_samples_processed})")
    
    with torch.no_grad():
        set_transformer_early_exit_mode(model, 'sft_teacher')
        
        ground_truth_answer = extract_solution(prompt_batch.ground_truth_answer, method="strict")
        #print(ground_truth_answer)
        
        #try up to 5 times total
        max_attempts = 5
        attempt = 0
        answer_correct = "no"
        
        while attempt < max_attempts and answer_correct == "no":
            attempt += 1
            
            sft_teacher_response, (sft_teacher_generated_tokens, sft_teacher_final_layer_logprobs, gathered_early_exit_hidden_states) = \
                generate_text(
                    model=model, 
                    prompt=prompt_batch.full_user_prompt, 
                    system_prompt=SYSTEM_PROMPT,
                    prefiller=PREFILLER, 
                    tokenizer=tokenizer, 
                    generation_config=config['generation'], 
                    device=device
                )
            #print(sft_teacher_response)
            predicted_answer = extract_answer_qwen(sft_teacher_response)
            #print(predicted_answer)
            
            #check if answer is correct
            if predicted_answer is not None and ground_truth_answer is not None:
                try:
                    pred_num = float(predicted_answer)
                    truth_num = float(ground_truth_answer)
                    if pred_num == truth_num:
                        answer_correct = "yes"
                        print(f"  Correct answer found on attempt {attempt}")
                        break
                except ValueError:
                    pass
            
            if attempt < max_attempts and answer_correct == "no":
                print(f"  Attempt {attempt} incorrect, retrying...")

                del sft_teacher_generated_tokens, sft_teacher_final_layer_logprobs, gathered_early_exit_hidden_states
                torch.cuda.empty_cache()
        
        if answer_correct == "no":
            print(f"  Failed to get correct answer after {max_attempts} attempts")
        
        #early exit probabilities on the final response
        early_output_log_probs = model.early_exit_hidden_state_readout(gathered_early_exit_hidden_states)
        
        #KL divergence calculations
        teacher_expanded = sft_teacher_final_layer_logprobs.unsqueeze(1).exp()
        early_output_probs = early_output_log_probs.exp()
        eps = 1e-16
        kl_div1 = - (teacher_expanded * (early_output_probs + eps).log()).sum(-1)
        kl_div2 = (teacher_expanded * ((teacher_expanded + eps) / (early_output_probs + eps)).log()).sum(-1)
        
        batch_data = {
            'batch_idx': batch_idx,
            'prompt_idx': prompt_batch.idx[0] if hasattr(prompt_batch, 'idx') else batch_idx,
            'full_user_prompt': prompt_batch.full_user_prompt,
            'ground_truth_answer': prompt_batch.ground_truth_answer,
            'predicted_answer': predicted_answer,
            'answer_correct': answer_correct,
            'attempts_needed': attempt,  #tracks how many attempts needed
            'solved_percentage': prompt_batch.difficulty,
            'difficulty_category': prompt_batch.difficulty_category,
            'sft_teacher_response': sft_teacher_response,  #final response used, either correct or final one
            'sft_teacher_generated_tokens': sft_teacher_generated_tokens.cpu(),
            'sft_teacher_final_layer_logprobs': sft_teacher_final_layer_logprobs.cpu(),
            'kl_div1_per_layer': kl_div1.cpu(),
            'kl_div2_per_layer': kl_div2.cpu(),
            'exitable_layer_idxs': model.exitable_layer_idxs.cpu(),
        }
        
        current_chunk_data.append(batch_data)
        total_samples_processed += 1
        
        if len(current_chunk_data) >= chunk_size:
            save_chunk(current_chunk_data, chunk_idx, output_dir)
            current_chunk_data = []
            chunk_idx += 1
        
        torch.cuda.empty_cache()

Processing batch 1/7473 (Total samples: 0)
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 74])
  Correct answer found on attempt 1
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 66])
  Attempt 1 incorrect, retrying...
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 66])
  Attempt 2 incorrect, retrying...
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 66])
  Correct answer found on attempt 3
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 96])
  Attempt 1 incorrect, retrying...
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 96])
  Attempt 2 incorrect, retrying...
full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 96])
  Correct answer found on attempt 3
full_tokenize currently only for Deepseek models!
prompt tokens shape: t

In [13]:
if current_chunk_data:
    print(f"\nSaving final chunk with {len(current_chunk_data)} samples...")
    save_chunk(current_chunk_data, chunk_idx, output_dir)


Saving final chunk with 19 samples...
Saved chunk 6 with 19 samples
  File: /workspace/data/teacher_generated_data_gzip/chunk_0006.pkl.gz
  Size: 11.80 MB


In [14]:
def merge_teacher_data_chunks(
    output_dir: str,
    merged_filename: str = "merged_teacher_data_sparse.pkl.gz",
    delete_chunks: bool = True,
    compresslevel: int = 9
):
    print(f"Merging chunks from {output_dir}...")

    #load metadata - now gzipped
    meta_path = os.path.join(output_dir, "metadata.pkl.gz")
    with gzip.open(meta_path, "rb") as f:
        metadata = pickle.load(f)

    chunk_files = sorted(
        f for f in os.listdir(output_dir)
        if f.startswith("chunk_") and f.endswith(".pkl.gz")
    )
    print(f"Found {len(chunk_files)} chunk files")

    merged_path = os.path.join(output_dir, merged_filename)
    total = 0

    with gzip.open(merged_path, "wb", compresslevel=compresslevel) as fout:
        # write header once
        pickle.dump({'metadata': metadata}, fout, protocol=pickle.HIGHEST_PROTOCOL)

        for i, cf in enumerate(chunk_files, 1):
            cpath = os.path.join(output_dir, cf)
            print(f"  [{i}/{len(chunk_files)}] {cf}")
            with gzip.open(cpath, "rb") as fin:
                chunk = pickle.load(fin)          # {'chunk_idx','data',...}
                for sample in chunk['data']:
                    pickle.dump(sample, fout, protocol=pickle.HIGHEST_PROTOCOL)
                total += len(chunk['data'])

            del chunk
            gc.collect()

            if delete_chunks:
                os.remove(cpath)
                print(f"    deleted {cf}")

        pickle.dump({'_end': True, 'num_samples': total}, fout, protocol=pickle.HIGHEST_PROTOCOL)

    size_mb = os.path.getsize(merged_path) / (1024 * 1024)
    print(f"\nTotal samples: {total}")
    print(f"Merged file size: {size_mb:.2f} MB")
    print(f"Saved to: {merged_path}")


def iter_merged_teacher_data(merged_path: str) -> Iterator[dict]:
    """
    Lazily iterate samples from the merged stream.
    Skips header and footer.
    """
    with gzip.open(merged_path, "rb") as f:
        header = pickle.load(f)  # {'metadata': ...}
        while True:
            try:
                obj = pickle.load(f)
            except EOFError:
                break
            if isinstance(obj, dict) and obj.get('_end'):
                break
            yield obj


In [9]:
def merge_teacher_data_chunks(output_dir, merged_filename="merged_teacher_data.pkl.gz"):
    """
    Merge all chunk files into a single file
    """
    print(f"Merging chunks from {output_dir}...")
    
    #load metadata - now gzipped
    metadata_path = os.path.join(output_dir, "metadata.pkl.gz")
    with gzip.open(metadata_path, 'rb') as f:
        metadata = pickle.load(f)
    
    #find all chunk files - now with .gz extension
    chunk_files = sorted([f for f in os.listdir(output_dir) if f.startswith("chunk_") and f.endswith(".pkl.gz")])
    print(f"Found {len(chunk_files)} chunk files to merge")
    
    all_data = []
    for chunk_file in chunk_files:
        chunk_path = os.path.join(output_dir, chunk_file)
        print(f"  Loading {chunk_file}...")
        with gzip.open(chunk_path, 'rb') as f:
            chunk_data = pickle.load(f)
            all_data.extend(chunk_data['data'])
            print(f"    Loaded {len(chunk_data['data'])} samples")
    
    #save merged data with gzip
    merged_path = os.path.join(output_dir, merged_filename)
    print(f"\nSaving merged data to {merged_path}...")
    with gzip.open(merged_path, 'wb', compresslevel=9) as f:
        pickle.dump({
            'teacher_data': all_data,
            'metadata': {
                **metadata,
                'num_samples': len(all_data),
                'num_chunks_merged': len(chunk_files)
            }
        }, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    file_size_mb = os.path.getsize(merged_path) / (1024 * 1024)
    print(f"Total samples: {len(all_data)}")
    print(f"Merged file size: {file_size_mb:.2f} MB")
    print(f"Saved to: {merged_path}")
    
    #return all_data

In [15]:
merge_teacher_data_chunks(output_dir)

Merging chunks from /workspace/data/teacher_generated_data_gzip...
Found 7 chunk files
  [1/7] chunk_0000.pkl.gz
    deleted chunk_0000.pkl.gz
  [2/7] chunk_0001.pkl.gz
    deleted chunk_0001.pkl.gz
  [3/7] chunk_0002.pkl.gz
    deleted chunk_0002.pkl.gz
  [4/7] chunk_0003.pkl.gz
    deleted chunk_0003.pkl.gz
  [5/7] chunk_0004.pkl.gz
    deleted chunk_0004.pkl.gz
  [6/7] chunk_0005.pkl.gz
    deleted chunk_0005.pkl.gz
  [7/7] chunk_0006.pkl.gz
    deleted chunk_0006.pkl.gz

Total samples: 619
Merged file size: 472.56 MB
Saved to: /workspace/data/teacher_generated_data_gzip/merged_teacher_data_sparse.pkl.gz


In [ ]:
def create_huggingface_datasets(merged_data_path, hf_dataset_name, push_to_hub=True):
    """
    Create and optionally push Hugging Face datasets from your split data
    """
    import random
    
    print("Loading merged dataset...")
    all_samples = list(iter_merged_teacher_data(merged_data_path))
    print(f"Total samples loaded: {len(all_samples)}")
    
    # Separate correct and incorrect answers
    correct_samples = []
    incorrect_samples = []
    
    for sample in all_samples:
        if sample.get('answer_correct') == 'yes':
            correct_samples.append(sample)
        else:
            incorrect_samples.append(sample)
    
    print(f"Correct answers: {len(correct_samples)}")
    print(f"Incorrect answers: {len(incorrect_samples)}")
    
    # Shuffle and split correct samples 50/50
    random.shuffle(correct_samples)
    mid_point = len(correct_samples) // 2
    sft_train = correct_samples[:mid_point]
    rl_train = correct_samples[mid_point:]
    
    print(f"SFT train: {len(sft_train)}")
    print(f"RL train: {len(rl_train)}")
    
    # Convert to DatasetDict format - keeping ALL fields
    def prepare_samples_for_hf(samples):
        """Convert samples to format suitable for HF datasets"""
        prepared = {
            'batch_idx': [],
            'prompt_idx': [],
            'question': [],
            'ground_truth_answer': [],
            'model_response': [],
            'predicted_answer': [],
            'answer_correct': [],
            'attempts_needed': [],
            'difficulty_category': [],
            'solved_percentage': [],
            'sft_teacher_generated_tokens': [],
            'sft_teacher_final_layer_logprobs': [],
            'kl_div1_per_layer': [],
            'kl_div2_per_layer': [],
            'exitable_layer_idxs': []
        }
        
        for sample in samples:
            prepared['batch_idx'].append(sample['batch_idx'])
            prepared['prompt_idx'].append(sample['prompt_idx'])
            prepared['question'].append(sample['full_user_prompt'][0])  # Extract from list
            prepared['ground_truth_answer'].append(sample['ground_truth_answer'])
            prepared['model_response'].append(sample['sft_teacher_response'])
            prepared['predicted_answer'].append(sample['predicted_answer'])
            prepared['answer_correct'].append(sample['answer_correct'])
            prepared['attempts_needed'].append(sample['attempts_needed'])
            prepared['difficulty_category'].append(sample['difficulty_category'])
            prepared['solved_percentage'].append(sample['solved_percentage'])
            prepared['sft_teacher_generated_tokens'].append(sample['sft_teacher_generated_tokens'])
            prepared['sft_teacher_final_layer_logprobs'].append(sample['sft_teacher_final_layer_logprobs'])
            prepared['kl_div1_per_layer'].append(sample['kl_div1_per_layer'])
            prepared['kl_div2_per_layer'].append(sample['kl_div2_per_layer'])
            prepared['exitable_layer_idxs'].append(sample['exitable_layer_idxs'])
        
        return prepared
    
    # Create datasets
    dataset_dict = DatasetDict({
        'total': Dataset.from_dict(prepare_samples_for_hf(all_samples)),
        'incorrect_answers': Dataset.from_dict(prepare_samples_for_hf(incorrect_samples)),
        'sft_train': Dataset.from_dict(prepare_samples_for_hf(sft_train)),
        'rl_train': Dataset.from_dict(prepare_samples_for_hf(rl_train))
    })
    
    # Print dataset info
    print("\nDataset splits created:")
    for split_name, dataset in dataset_dict.items():
        print(f"{split_name}: {len(dataset)} samples")
        print(f"  Columns: {dataset.column_names}")
    
    # Save locally first
    dataset_dict.save_to_disk(f"./{hf_dataset_name}")
    print(f"Saved locally to ./{hf_dataset_name}")
    
    # Push to hub if requested
    if push_to_hub:
        print(f"Pushing to Hugging Face Hub as {hf_dataset_name}...")
        dataset_dict.push_to_hub(hf_dataset_name)
        print(f"Dataset uploaded to: https://huggingface.co/datasets/{hf_dataset_name}")
    
    return dataset_dict

In [ ]:
datasets = create_huggingface_datasets(
    merged_data_path=os.path.join(output_dir, "merged_teacher_data_sparse.pkl.gz"),  # Path to your merged file
    hf_dataset_name="lizardp1/gsm8k-qwen-early-exit",  # Your HF dataset name
    push_to_hub=True  # Set to False if you only want to save locally first
)

In [28]:
os.path.getsize(os.path.join(output_dir, "merged_teacher_data_sparse_combtensor.pkl.gz")) / (1024 * 1024)

26.803590774536133

In [29]:
os.path.getsize(os.path.join(output_dir, "merged_teacher_data_sparse.pkl.gz")) / (1024 * 1024)

26.832829475402832

In [30]:
os.path.getsize(os.path.join(output_dir, "merged_teacher_data_original.pkl.gz")) / (1024 * 1024)

2120.7067728042603